# Lab 10 - Generación de Datos Sintéticos con SDV

**Librería SDV (Synthetic Data Vault)**

* Autor: Jose Villalobos
* Fecha: 04 de junio

In [ ]:
!pip install sdv sdmetrics

In [ ]:
#Importamos librerias
import pandas as pd
import numpy as np
import random

In [ ]:
import sdv  # Generar datos sintéticos
import sdmetrics

from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

In [ ]:
from sdmetrics.reports.single_table import QualityReport

In [ ]:
print("SDV:", sdv.__version__)

## Crear DataFrame de ejemplo

In [ ]:
# Creamos un DataFrame a partir de datos base
dfClientes = pd.DataFrame(
    {
        "cliente_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        "edad": [23, 33, 43, 28, 53, 56, 43, 56, 65, 40],
        "ingreso_mensual": [25000, 15000, 20000, 10000, 5000, 17000, 30000, 12000, 35000, 75000],
        "ciudad": ["Veracruz", "Cordoba", "Paso del macho", "Amatlan", "Fortin",
                   "Cuitlahuac", "Yanga", "Cordoba", "Cuitlahuac", "Fortin"]
    }
)

In [ ]:
# Visualizamos las primeras filas para inspeccionar la estructura de los datos
dfClientes.head()

## Definir metadatos y entrenar modelo

In [ ]:
#Definir los metadatos
metadata = SingleTableMetadata()

In [ ]:
metadata.detect_from_dataframe(
    data=dfClientes
)

In [ ]:
metadata.to_dict()

In [ ]:
#Guardamos el metadata en un archivo JSON
metadata.save_to_json(
    "dfClientes_metadata.json"
)

In [ ]:
#Entrenamos el modelo para generar los datos sintéticos
synthesizer = GaussianCopulaSynthesizer(
    metadata
)

In [ ]:
#Entrenamiento
synthesizer.fit(
    dfClientes
)

## Generar datos sintéticos

In [ ]:
#Generamos los datos sintéticos
clientes_sinteticos = synthesizer.sample(
    num_rows=100
)

In [ ]:
# Visualizamos las primeras filas para inspeccionar la estructura de los datos
clientes_sinteticos.head()

In [ ]:
# Obtenemos un resumen de las columnas, tipos de datos y valores no nulos
clientes_sinteticos.info()

In [ ]:
# Calculamos estadísticas descriptivas básicas (media, min, max, etc.)
clientes_sinteticos.describe(include="all")

## Contaminación de datos (GIGO)

In [ ]:
#DataFrame contaminado
dfClientesGIGO = clientes_sinteticos.copy()

In [ ]:
#Colocar edades imposibles
indices = random.sample(list(dfClientesGIGO.index), 5)

In [ ]:
dfClientesGIGO.loc[indices, "edad"] = -5

In [ ]:
#Introducimos valores duplicados
duplicados = dfClientesGIGO.sample(10, random_state=42)

In [ ]:
dfClientesGIGO = pd.concat([dfClientesGIGO, duplicados], ignore_index=True)

In [ ]:
#Verificamos valores duplicados
dfClientesGIGO.duplicated().sum()

In [ ]:
# Calculamos estadísticas descriptivas básicas (media, min, max, etc.)
dfClientesGIGO.describe(include="all")

## Reporte de calidad

In [ ]:
#Generamos el reporte
reporte = QualityReport()

In [ ]:
reporte.generate(
    real_data=dfClientes,
    synthetic_data=clientes_sinteticos,
    metadata=metadata.to_dict()
)

In [ ]:
#Reporte con datos contaminados
reporte_gigo = QualityReport()
reporte_gigo.generate(
    real_data=dfClientes,
    synthetic_data=dfClientesGIGO,
    metadata=metadata.to_dict()
)